In [2]:
import random
print(random.randint(2, 11))

8


In [ ]:
# from __future__ import annotations

from qiskit import QuantumCircuit
from qiskit.circuit import Gate


def bernstein_vazirani_oracle(n: int, secret_string: str) -> Gate:
    """Generates the oracle for the Bernstein-Vazirani algorithm."""
    oracle = QuantumCircuit(n + 1)
    s = secret_string[::-1]
    for qubit in range(n):
        if s[qubit] == "1":
            oracle.cx(qubit, n)
    oracle_gate = oracle.to_gate()
    oracle_gate.name = "Oracle"
    return oracle_gate


def oracle_gate_to_qasm3_gate_def(
    oracle_gate: Gate,
    gate_name: str | None = None,
    qubit_prefix: str = "_gate_q_",
    indent: str = "  ",
) -> str:
    """
    Convert a Qiskit Gate (with a definition) into a standalone OpenQASM 3.0 gate definition string.

    This is intentionally lightweight and supports the BV oracle produced above (only 'cx').
    Extend the 'emit_instruction' mapping if you add more operations to the oracle.
    """
    if oracle_gate.definition is None:
        raise ValueError("oracle_gate has no definition. Make sure it was created via circuit.to_gate().")

    qc: QuantumCircuit = oracle_gate.definition
    name = gate_name or getattr(oracle_gate, "name", None) or "Oracle"

    # Gate argument names: _gate_q_0, _gate_q_1, ...
    arg_names = [f"{qubit_prefix}{i}" for i in range(oracle_gate.num_qubits)]
    header = f"gate {name} {', '.join(arg_names)} {{\n"

    # Map each Qiskit qubit object in the definition to an integer index
    qb_to_idx = {qb: i for i, qb in enumerate(qc.qubits)}

    def emit_instruction(inst) -> str:
        op = inst.operation
        qargs = inst.qubits

        # BV oracle uses only CX; keep it strict so mismatches fail loudly.
        if op.name in ("cx", "cx_gate"):
            c = arg_names[qb_to_idx[qargs[0]]]
            t = arg_names[qb_to_idx[qargs[1]]]
            return f"cx {c}, {t};"

        raise NotImplementedError(
            f"Unsupported operation in oracle gate definition: {op.name!r}. "
            f"Extend emit_instruction() to handle it."
        )

    body_lines = [indent + emit_instruction(inst) for inst in qc.data]
    footer = "}\n"

    return header + ("\n".join(body_lines) + ("\n" if body_lines else "")) + footer


# ------------------ Example usage ------------------
if __name__ == "__main__":
    n = 5
    secret = "10111"  # example
    oracle_gate = bernstein_vazirani_oracle(n, secret)

    qasm3_gate_def = oracle_gate_to_qasm3_gate_def(oracle_gate)
    print(qasm3_gate_def)
